# AHAAR — Ingredient Detector (clean, end-to-end)

Trains YOLOv11m on all pantry ingredient classes merged from three sources:
- 36 produce (Kaggle fruit-veg)
- 19 Indian spices (Kaggle indian-spices)
- 55 scraped raw ingredients (from your Google Drive)

**Before Run All:**
1. `Runtime → Change runtime type → T4 GPU`
2. 🔑 Colab secrets: `KAGGLE_USERNAME`, `KAGGLE_KEY`, `HF_TOKEN` *(HF optional)*
3. Your `scraped.zip` must be at the Drive path configured in cell 2.

Then `Runtime → Run all`. Cells 1–7 take < 15 min and gate the long training in cell 8.

## 1. Install

In [ ]:
!pip install -q ultralytics==8.3.* huggingface_hub==0.26.* kaggle six onnx onnxruntime pyyaml 2>&1 | tail -3
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime → Change runtime type → T4 GPU, then re-run.'
print('GPU:', torch.cuda.get_device_name(0),
      round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

## 2. Config

In [ ]:
from pathlib import Path

REPO_URL    = 'https://github.com/workspace220106/Aahar-Kitchen-System.git'
REPO_BRANCH = 'main'

# Where your scraped.zip lives on Google Drive
DRIVE_SCRAPED_PATH = '/content/drive/MyDrive/AAhar/scraped/scraped.zip'

# Where training outputs land (Drive survives Colab disconnects)
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/AAhar/runs'

MODEL     = 'yolo11m.pt'
IMG_SIZE  = 640
EPOCHS    = 60
BATCH     = 16
VAL_SPLIT = 0.10

HF_MODEL_REPO = 'rajivagarwal220106/aahar-ingredient-detector'

print('Config OK.')

## 3. Clone repo + Kaggle creds + load manifest

In [ ]:
import os, json, subprocess
from google.colab import userdata

os.chdir('/content')
if not Path('/content/aahar').exists():
    subprocess.run(['git','clone','--branch',REPO_BRANCH,'--depth','1',REPO_URL,
                    '/content/aahar'], check=True)

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
Path('/root/.kaggle').mkdir(exist_ok=True)
Path('/root/.kaggle/kaggle.json').write_text(json.dumps({
    'username': os.environ['KAGGLE_USERNAME'],
    'key':      os.environ['KAGGLE_KEY'],
}))
os.chmod('/root/.kaggle/kaggle.json', 0o600)

manifest = json.load(open('/content/aahar/datasets/processed/class_manifest.json'))
TARGET_CLASSES = sorted(set(manifest['ingredient_training_classes']))
print(f'Target ingredient classes: {len(TARGET_CLASSES)}')

## 4. Mount Drive and extract scraped.zip (handles Windows PowerShell bug)

Windows' `Compress-Archive` encodes paths with backslashes; Linux unzip treats `\` as a filename character. This cell autodetects that case and reorganises the flat files into the proper folder tree.

In [ ]:
import shutil, zipfile
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

base = Path(DRIVE_SCRAPED_PATH)
if base.is_file() and base.suffix.lower() == '.zip':
    src = base
elif base.is_dir():
    zs = sorted(base.glob('*.zip'))
    src = zs[0] if zs else base
elif Path(str(base) + '.zip').exists():
    src = Path(str(base) + '.zip')
else:
    raise FileNotFoundError(f'Nothing at {base}')

STAGE = Path('/content/raw/scraped_in')
DST   = Path('/content/raw/scraped')
for d in (STAGE, DST):
    if d.exists(): shutil.rmtree(d)
    d.mkdir(parents=True)

if src.is_file():
    print(f'Unzipping {src} ({src.stat().st_size/1e6:.0f} MB)')
    with zipfile.ZipFile(src) as z: z.extractall(STAGE)
else:
    shutil.copytree(src, STAGE, dirs_exist_ok=True)

# Windows backslash case: flat files with '\' in name
flat = [f for f in os.listdir(STAGE)
        if os.path.isfile(os.path.join(STAGE, f)) and '\\' in f]
if flat:
    print(f'Windows-backslash zip detected; reorganising {len(flat)} files...')
    for f in flat:
        parts = f.split('\\')
        if parts[0].lower() == 'scraped':
            parts = parts[1:]
        if not parts: continue
        tgt = DST.joinpath(*parts)
        tgt.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(os.path.join(STAGE, f), tgt)
else:
    # Normal zip: walk STAGE for deepest dir of class folders
    def score(d):
        if not d.is_dir(): return 0
        subs = [x for x in d.iterdir() if x.is_dir() and not x.name.startswith('__')]
        return sum(1 for sd in subs
                   if any(p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}
                          for p in sd.rglob('*') if p.is_file()))
    best, bs = STAGE, score(STAGE)
    for p in STAGE.rglob('*'):
        if p.is_dir():
            s = score(p)
            if s > bs: best, bs = p, s
    for item in best.iterdir():
        shutil.move(str(item), DST / item.name)

SCRAPED_ROOT = DST
class_dirs = [d for d in SCRAPED_ROOT.iterdir() if d.is_dir()]
print(f'Scraped class folders: {len(class_dirs)}')
assert len(class_dirs) >= 40, f'Only {len(class_dirs)} scraped class folders'

## 5. Fetch Kaggle ingredient datasets (~3.3 GB)

In [ ]:
RAW = Path('/content/raw'); RAW.mkdir(exist_ok=True)
for ref, sub in [
    ('kritikseth/fruit-and-vegetable-image-recognition', 'fruit-veg'),
    ('tankitrathaur/indian-spices-images-data',          'indian-spices'),
]:
    dst = RAW / sub
    if dst.exists() and any(dst.iterdir()):
        print(f'SKIP (exists): {ref}')
        continue
    dst.mkdir(parents=True, exist_ok=True)
    print(f'--- {ref} ---')
    subprocess.run(['kaggle','datasets','download','-d',ref,'-p',str(dst),'--unzip'],
                   check=True)
subprocess.run(['du','-sh',str(RAW/'fruit-veg'), str(RAW/'indian-spices')], check=True)

## 6. Build YOLO layout

Merges all 3 sources against the 110-class manifest. Matches every source-folder name against 4 canonical variants so `toor_dal`, `Bay Leaf`, `curry leaves`, etc. all map correctly.

In [ ]:
import re, random
import yaml
from PIL import Image
from collections import defaultdict

ALIASES = {
    'carrots':'carrot','raddish':'radish','sweetpotato':'sweet potato',
    'sweetcorn':'sweet corn','jalepeno':'jalapeno','cinnamom stick':'cinnamon stick',
    'grapes':'grape',
}
def canon(s):
    s = s.strip().lower().replace('-','').replace('_',' ')
    s = re.sub(r'\s+',' ', s)
    return ALIASES.get(s, s)
def snake(s):
    return re.sub(r'\s+','_', s.strip().lower().replace('-','_'))

# Build variant map: any of 4 forms → canonical target name
target_variants = {}
for name in TARGET_CLASSES:
    for v in {name, canon(name), snake(name), snake(canon(name))}:
        target_variants[v] = name

pool = defaultdict(list)
def ingest(root: Path):
    if not root.exists(): return
    for d in root.iterdir():
        if not d.is_dir() or d.name.startswith('__') or 'MetaData' in d.name: continue
        for v in (d.name, canon(d.name), snake(d.name), snake(canon(d.name))):
            if v in target_variants:
                tgt = target_variants[v]
                for p in d.rglob('*'):
                    if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}:
                        pool[tgt].append(p)
                break

for split in ('train','validation','test'):
    ingest(Path(f'/content/raw/fruit-veg/{split}'))
ingest(Path('/content/raw/indian-spices/indian-spices-image-data'))
ingest(SCRAPED_ROOT)

class_names = sorted(pool.keys())
cls_id = {n:i for i,n in enumerate(class_names)}
total = sum(len(v) for v in pool.values())
missing = sorted(set(TARGET_CLASSES) - set(class_names))
print(f'Classes kept: {len(class_names)} / {len(TARGET_CLASSES)} target')
print(f'Total images: {total}')
if missing: print(f'Missing: {missing}')

DATA = Path('/content/aahar/data')
for sub in ('images/train','images/val','labels/train','labels/val'):
    d = DATA / sub
    if d.exists(): shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

random.seed(42)
for name, imgs in pool.items():
    random.shuffle(imgs)
    n_val = max(1, int(len(imgs)*VAL_SPLIT))
    for i, img in enumerate(imgs):
        split = 'val' if i < n_val else 'train'
        try:
            with Image.open(img) as im: im.verify()
        except Exception: continue
        new = f'{cls_id[name]:03d}_{i:06d}{img.suffix.lower()}'
        shutil.copy2(img, DATA / f'images/{split}' / new)
        (DATA / f'labels/{split}' / new.rsplit('.',1)[0]).with_suffix('.txt')\
            .write_text(f'{cls_id[name]} 0.5 0.5 1.0 1.0\n')

cfg_path = Path('/content/aahar/aahar.yaml')
cfg_path.write_text(yaml.safe_dump({
    'path': str(DATA), 'train': 'images/train', 'val': 'images/val',
    'nc': len(class_names), 'names': class_names,
}, sort_keys=False))
print(f'\ntrain: {len(list((DATA/"images/train").iterdir()))}  '
      f'val: {len(list((DATA/"images/val").iterdir()))}')

## 7. Sanity check — stop here if anything looks wrong

Do not run cell 8 until this prints ≥ 100 classes.

In [ ]:
from collections import Counter
cfg = yaml.safe_load(open(cfg_path))
print('nc:', cfg['nc'])
counts = Counter()
for f in (DATA/'images/train').iterdir():
    counts[int(f.name.split('_')[0])] += 1
print(f'Per-class train counts (min/mean/max): '
      f'{min(counts.values())}/{sum(counts.values())//len(counts)}/{max(counts.values())}')
print('10 classes with fewest images:')
light = sorted([(n, counts[cls_id[n]]) for n in class_names], key=lambda x:x[1])[:10]
for n, c in light: print(f'  {n:25s} {c}')
if cfg['nc'] < 100:
    print('\n❌ ABORT: fewer than 100 classes. Do NOT run cell 8.')
else:
    print('\n✅ OK — safe to run cell 8.')

## 8. Train (~1.5–2 hrs on T4) — resumable

If Colab disconnects, re-run this cell: it resumes from the latest `last.pt` saved on Drive.

In [ ]:
from ultralytics import YOLO

out = Path(DRIVE_OUTPUT_DIR); out.mkdir(parents=True, exist_ok=True)
last_pt = out / 'aahar_ingredients_v1' / 'weights' / 'last.pt'

if last_pt.exists():
    print(f'Resuming from {last_pt}')
    model = YOLO(str(last_pt))
    results = model.train(resume=True)
else:
    model = YOLO(MODEL)
    results = model.train(
        data=str(cfg_path), epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH,
        device=0, project=str(out), name='aahar_ingredients_v1',
        patience=15, cache='ram', amp=True, save=True, save_period=5,
    )

best = out / 'aahar_ingredients_v1' / 'weights' / 'best.pt'
print('Best:', best, 'exists:', best.exists())

## 9. Export ONNX + TFLite + push to Hugging Face

In [ ]:
from ultralytics import YOLO
trained = YOLO(str(best))
onnx_path   = trained.export(format='onnx',  imgsz=IMG_SIZE, opset=12, dynamic=False, simplify=True)
tflite_path = trained.export(format='tflite', imgsz=IMG_SIZE, int8=False)
print('ONNX :', onnx_path)
print('TFLite:', tflite_path)

try:
    from huggingface_hub import HfApi, create_repo
    token = userdata.get('HF_TOKEN')
    if token:
        create_repo(HF_MODEL_REPO, token=token, exist_ok=True, repo_type='model')
        api = HfApi(token=token)
        for p in [best, Path(onnx_path), Path(tflite_path), cfg_path]:
            api.upload_file(path_or_fileobj=str(p), path_in_repo=p.name,
                            repo_id=HF_MODEL_REPO, token=token)
        print(f'Uploaded → https://huggingface.co/{HF_MODEL_REPO}')
    else:
        print('HF_TOKEN not set; skipping upload.')
except Exception as e:
    print('HF upload failed:', e)